# MINOR PROJECT
# NAME : S.NAVINKUMAR

In [7]:
import re
import numpy as np
from collections import defaultdict, Counter
from datetime import datetime
from google.colab import files

In [8]:
uploaded = files.upload()

filename = next(iter(uploaded))

with open(filename, "r", encoding="utf-8") as file:
    lines = file.readlines()

print(f"Loaded {len(lines)} lines successfully.")

Saving hostel_bois.txt to hostel_bois (1).txt
Loaded 3178 lines successfully.


In [9]:
messages = []

system_messages = 0

participants = set()

message_count = defaultdict(int)
word_count = defaultdict(int)
media_count = defaultdict(int)
deleted_count = defaultdict(int)

hour_count = defaultdict(int)
day_count = defaultdict(int)
weekday_count = defaultdict(int)
month_count = defaultdict(int)

active_dates = defaultdict(set)
message_length = defaultdict(list)

hour_day_matrix = np.zeros((24, 7), dtype=int)

In [10]:
pattern = r"^(\d{2}/\d{2}/\d{2}), (\d{2}:\d{2}) - (.*)"

In [11]:
for line in lines:

    line = line.strip()

    if not line:
        continue

    match = re.match(pattern, line)

    if not match:
        continue

    date, time, content = match.groups()

    if ":" not in content:
        system_messages += 1
        continue

    sender, message = content.split(":", 1)

    sender = sender.strip()
    message = message.strip()

    participants.add(sender)

    dt = datetime.strptime(date + " " + time, "%d/%m/%y %H:%M")

    messages.append({
        "date": dt.date(),
        "time": dt.time(),
        "hour": dt.hour,
        "weekday": dt.strftime("%A"),
        "month": dt.strftime("%B"),
        "sender": sender,
        "message": message
    })

print("Messages Parsed:", len(messages))
print("System Messages:", system_messages)
print("Participants:", participants)

Messages Parsed: 3174
System Messages: 4
Participants: {'Karan', 'Priya', 'Neha', 'Vikas', 'Aman', 'Rahul'}


PART - 2

In [12]:
for msg in messages:

    sender = msg["sender"]
    text = msg["message"]

    # Message Count
    message_count[sender] += 1

    # Active Dates
    active_dates[sender].add(msg["date"])

    # Time Analysis
    hour_count[msg["hour"]] += 1
    weekday_count[msg["weekday"]] += 1
    month_count[msg["month"]] += 1

    # NumPy Heatmap
    hour_day_matrix[msg["hour"]][msg["date"].weekday()] += 1

    # Media
    if "<Media omitted>" in text:
        media_count[sender] += 1
        continue

    # Deleted
    if "This message was deleted" in text:
        deleted_count[sender] += 1
        continue

    # Message Length
    message_length[sender].append(len(text))

    # Word Count
    words = text.lower().split()
    word_count[sender] += len(words)

In [13]:
print("="*60)
print("TOP ACTIVE USERS")
print("="*60)

rank = sorted(message_count.items(),
              key=lambda x:x[1],
              reverse=True)

for i,(user,count) in enumerate(rank,1):
    print(f"{i}. {user:10} {count}")

TOP ACTIVE USERS
1. Rahul      953
2. Priya      718
3. Neha       635
4. Aman       490
5. Karan      354
6. Vikas      24


In [14]:
print("="*60)
print("AVERAGE MESSAGE LENGTH")
print("="*60)

for user in sorted(participants):

    if len(message_length[user]) == 0:
        avg = 0
    else:
        avg = sum(message_length[user])/len(message_length[user])

    print(f"{user:10} {avg:.2f} characters")

AVERAGE MESSAGE LENGTH
Aman       26.72 characters
Karan      311.41 characters
Neha       25.37 characters
Priya      28.21 characters
Rahul      10.69 characters
Vikas      7.23 characters


In [15]:
print("="*60)
print("TOTAL WORDS")
print("="*60)

ranking = sorted(word_count.items(),
                 key=lambda x:x[1],
                 reverse=True)

for user,words in ranking:
    print(f"{user:10} {words}")

TOTAL WORDS
Karan      19681
Priya      3560
Neha       3317
Aman       2430
Rahul      2399
Vikas      40


In [16]:
print("="*60)
print("MEDIA SHARED")
print("="*60)

for user in sorted(participants):
    print(f"{user:10} {media_count[user]}")

MEDIA SHARED
Aman       4
Karan      7
Neha       8
Priya      4
Rahul      7
Vikas      2


In [17]:
print("="*60)
print("DELETED MESSAGES")
print("="*60)

for user in sorted(participants):
    print(f"{user:10} {deleted_count[user]}")

DELETED MESSAGES
Aman       2
Karan      2
Neha       3
Priya      2
Rahul      6
Vikas      0


In [18]:
print("="*60)
print("ACTIVE DAYS")
print("="*60)

for user in sorted(participants):
    print(f"{user:10} {len(active_dates[user])}")

ACTIVE DAYS
Aman       60
Karan      60
Neha       60
Priya      60
Rahul      60
Vikas      16


In [19]:
print("="*60)
print("SPAMMER RANKING")
print("="*60)

spam = sorted(message_count.items(),
              key=lambda x:x[1],
              reverse=True)

for i,(user,count) in enumerate(spam,1):
    print(f"{i}. {user} ({count} messages)")

SPAMMER RANKING
1. Rahul (953 messages)
2. Priya (718 messages)
3. Neha (635 messages)
4. Aman (490 messages)
5. Karan (354 messages)
6. Vikas (24 messages)


In [20]:
print("="*60)
print("GHOST RANKING")
print("="*60)

ghost = sorted(message_count.items(),
               key=lambda x:x[1])

for i,(user,count) in enumerate(ghost,1):
    print(f"{i}. {user} ({count} messages)")

GHOST RANKING
1. Vikas (24 messages)
2. Karan (354 messages)
3. Aman (490 messages)
4. Neha (635 messages)
5. Priya (718 messages)
6. Rahul (953 messages)


In [21]:
print("="*60)
print("LONGEST AVERAGE MESSAGE")
print("="*60)

avg_length = {}

for user in participants:

    if len(message_length[user]) == 0:
        avg_length[user] = 0
    else:
        avg_length[user] = sum(message_length[user])/len(message_length[user])

ranking = sorted(avg_length.items(),
                 key=lambda x:x[1],
                 reverse=True)

for user,length in ranking:
    print(f"{user:10} {length:.2f}")

LONGEST AVERAGE MESSAGE
Karan      311.41
Priya      28.21
Aman       26.72
Neha       25.37
Rahul      10.69
Vikas      7.23


In [22]:
print("="*60)
print("MOST ACTIVE HOURS")
print("="*60)

hours = sorted(hour_count.items(),
               key=lambda x:x[1],
               reverse=True)

for hour,count in hours:

    print(f"{hour:02d}:00 - {count}")

MOST ACTIVE HOURS
18:00 - 248
19:00 - 228
12:00 - 193
16:00 - 189
21:00 - 177
17:00 - 173
20:00 - 166
14:00 - 162
10:00 - 160
13:00 - 159
23:00 - 159
09:00 - 151
15:00 - 131
08:00 - 122
22:00 - 116
11:00 - 114
04:00 - 110
02:00 - 83
01:00 - 82
03:00 - 77
00:00 - 57
07:00 - 55
06:00 - 33
05:00 - 29


In [23]:
print("="*60)
print("WEEKDAY ACTIVITY")
print("="*60)

days = sorted(weekday_count.items(),
              key=lambda x:x[1],
              reverse=True)

for day,count in days:
    print(f"{day:10} {count}")

WEEKDAY ACTIVITY
Wednesday  483
Monday     482
Thursday   475
Saturday   459
Sunday     446
Tuesday    426
Friday     403


In [24]:
print("="*60)
print("MONTHLY ACTIVITY")
print("="*60)

for month,count in month_count.items():
    print(f"{month:10} {count}")

MONTHLY ACTIVITY
April      1582
May        1592


In [25]:
all_words = []

for msg in messages:

    text = msg["message"]

    if "<Media omitted>" in text:
        continue

    if "This message was deleted" in text:
        continue

    words = text.lower().split()

    for word in words:

        word = word.strip(".,!?()[]{}\"'")

        if len(word) > 2:
            all_words.append(word)

counter = Counter(all_words)

print("="*60)
print("TOP 20 WORDS")
print("="*60)

for word,count in counter.most_common(20):
    print(f"{word:20} {count}")

TOP 20 WORDS
the                  1494
and                  1013
was                  385
how                  321
guys                 318
you                  312
this                 311
about                274
for                  270
hai                  268
today                257
his                  217
that                 211
have                 209
just                 208
which                202
everyone             187
telling              179
from                 174
bhai                 160


PART - 3

In [26]:
print("="*75)
print("24 x 7 ACTIVITY HEATMAP")
print("="*75)

days = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]

print("Hour ", end="")

for d in days:
    print(f"{d:^8}", end="")

print()

for hour in range(24):

    print(f"{hour:02d}:00", end=" ")

    for day in range(7):

        print(f"{hour_day_matrix[hour][day]:8}", end="")

    print()

24 x 7 ACTIVITY HEATMAP
Hour   Mon     Tue     Wed     Thu     Fri     Sat     Sun   
00:00        8      11       6       7       5      11       9
01:00       17      16      11       5      11      15       7
02:00       13      13      12      15      17       9       4
03:00       11      25       6      14       4      10       7
04:00       11      14      11       9      22      17      26
05:00        0      12       4       6       3       0       4
06:00        4       4       0       1       6       3      15
07:00       10       9       8       9       4       7       8
08:00       18      18      20      16      21      14      15
09:00       12      21      31      23      26      21      17
10:00       18      19      25      20      23      23      32
11:00       15       7      15      30      11      14      22
12:00       31      21      27      37      23      20      34
13:00       30      18      17      24      25      22      23
14:00       18      20      37  

In [27]:
daily_messages = defaultdict(int)

for msg in messages:
    daily_messages[msg["date"]] += 1

print("="*75)
print("TOP 15 BUSIEST DAYS")
print("="*75)

top_days = sorted(
    daily_messages.items(),
    key=lambda x:x[1],
    reverse=True
)

for day,count in top_days[:15]:
    print(day,"->",count)

TOP 15 BUSIEST DAYS
2024-05-04 -> 76
2024-04-24 -> 72
2024-04-28 -> 72
2024-04-03 -> 71
2024-04-27 -> 65
2024-05-26 -> 64
2024-05-09 -> 63
2024-05-17 -> 62
2024-05-23 -> 62
2024-04-06 -> 60
2024-04-15 -> 59
2024-05-05 -> 59
2024-04-22 -> 58
2024-04-25 -> 58
2024-05-18 -> 58


In [28]:
print("="*75)
print("HOURLY ACTIVITY")
print("="*75)

for hour,count in sorted(hour_count.items()):

    bar = "█"*(count//10)

    print(f"{hour:02d}:00 | {bar} {count}")

HOURLY ACTIVITY
00:00 | █████ 57
01:00 | ████████ 82
02:00 | ████████ 83
03:00 | ███████ 77
04:00 | ███████████ 110
05:00 | ██ 29
06:00 | ███ 33
07:00 | █████ 55
08:00 | ████████████ 122
09:00 | ███████████████ 151
10:00 | ████████████████ 160
11:00 | ███████████ 114
12:00 | ███████████████████ 193
13:00 | ███████████████ 159
14:00 | ████████████████ 162
15:00 | █████████████ 131
16:00 | ██████████████████ 189
17:00 | █████████████████ 173
18:00 | ████████████████████████ 248
19:00 | ██████████████████████ 228
20:00 | ████████████████ 166
21:00 | █████████████████ 177
22:00 | ███████████ 116
23:00 | ███████████████ 159


In [29]:
print("="*75)
print("WEEKDAY ACTIVITY")
print("="*75)

order=[
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

for day in order:

    count=weekday_count[day]

    bar="█"*(count//15)

    print(f"{day:10} {bar} {count}")


WEEKDAY ACTIVITY
Monday     ████████████████████████████████ 482
Tuesday    ████████████████████████████ 426
Wednesday  ████████████████████████████████ 483
Thursday   ███████████████████████████████ 475
Friday     ██████████████████████████ 403
Saturday   ██████████████████████████████ 459
Sunday     █████████████████████████████ 446


In [30]:
print("="*75)
print("USER SCORECARD")
print("="*75)

for user in sorted(participants):

    avg=0

    if len(message_length[user])>0:
        avg=sum(message_length[user])/len(message_length[user])

    print()

    print(user)

    print("-"*30)

    print("Messages :",message_count[user])

    print("Words :",word_count[user])

    print("Media :",media_count[user])

    print("Deleted :",deleted_count[user])

    print("Average Length :",round(avg,2))

    print("Active Days :",len(active_dates[user]))

USER SCORECARD

Aman
------------------------------
Messages : 490
Words : 2430
Media : 4
Deleted : 2
Average Length : 26.72
Active Days : 60

Karan
------------------------------
Messages : 354
Words : 19681
Media : 7
Deleted : 2
Average Length : 311.41
Active Days : 60

Neha
------------------------------
Messages : 635
Words : 3317
Media : 8
Deleted : 3
Average Length : 25.37
Active Days : 60

Priya
------------------------------
Messages : 718
Words : 3560
Media : 4
Deleted : 2
Average Length : 28.21
Active Days : 60

Rahul
------------------------------
Messages : 953
Words : 2399
Media : 7
Deleted : 6
Average Length : 10.69
Active Days : 60

Vikas
------------------------------
Messages : 24
Words : 40
Media : 2
Deleted : 0
Average Length : 7.23
Active Days : 16


In [31]:
night_messages=defaultdict(int)

for msg in messages:

    h=msg["hour"]

    if h>=23 or h<=4:

        night_messages[msg["sender"]]+=1

night_percentage={}

for user in participants:

    if message_count[user]==0:

        night_percentage[user]=0

    else:

        night_percentage[user]=round(
            night_messages[user]/message_count[user]*100,
            2
        )

In [33]:
long_messages=defaultdict(int)

for msg in messages:

    if len(msg["message"])>120:

        long_messages[msg["sender"]]+=1

long_percent={}

for user in participants:

    if message_count[user]==0:

        long_percent[user]=0

    else:

        long_percent[user]=round(
            long_messages[user]/message_count[user]*100,
            2
        )

In [51]:
personality={}

highest=max(message_count.values())

lowest=min(message_count.values())

for user in participants:

    if message_count[user]==highest:

        personality[user]=" Spammer"

    elif message_count[user]==lowest:

        personality[user]=" Ghost"

    elif night_percentage[user]>60:

        personality[user]=" Night Owl"

    elif long_percent[user]>40:

        personality[user]="Story Teller"

    elif media_count[user]>8:

        personality[user]=" Meme Lord"

    elif deleted_count[user]>3:

        personality[user]=" Secret Agent"

    elif avg_length:=(
        sum(message_length[user])/len(message_length[user])
        if len(message_length[user])>0
        else 0
    )<20:

        personality[user]="⚡ One Liner"

    else:

        personality[user]=" Balanced Human"

In [52]:
print("="*75)
print("GROUP PERSONALITY REPORT")
print("="*75)

for user in sorted(participants):

    print()

    print(user)

    print("-"*25)

    print("Role :",personality[user])

    print("Messages :",message_count[user])

    print("Night % :",night_percentage[user])

    print("Words :",word_count[user])

    print("Media :",media_count[user])

    print("Deleted :",deleted_count[user])

    print("Days Active :",len(active_dates[user]))

GROUP PERSONALITY REPORT

Aman
-------------------------
Role :  Night Owl
Messages : 490
Night % : 79.8
Words : 2430
Media : 4
Deleted : 2
Days Active : 60

Karan
-------------------------
Role : Story Teller
Messages : 354
Night % : 2.26
Words : 19681
Media : 7
Deleted : 2
Days Active : 60

Neha
-------------------------
Role :  Balanced Human
Messages : 635
Night % : 4.72
Words : 3317
Media : 8
Deleted : 3
Days Active : 60

Priya
-------------------------
Role :  Balanced Human
Messages : 718
Night % : 1.25
Words : 3560
Media : 4
Deleted : 2
Days Active : 60

Rahul
-------------------------
Role :  Spammer
Messages : 953
Night % : 13.43
Words : 2399
Media : 7
Deleted : 6
Days Active : 60

Vikas
-------------------------
Role :  Ghost
Messages : 24
Night % : 8.33
Words : 40
Media : 2
Deleted : 0
Days Active : 16


In [36]:
print("="*75)
print("GROUP SUMMARY")
print("="*75)

print("Participants :",len(participants))

print("Messages :",len(messages))

print("System Messages :",system_messages)

print("Media Shared :",sum(media_count.values()))

print("Deleted Messages :",sum(deleted_count.values()))

print("Unique Words :",len(set(all_words)))

GROUP SUMMARY
Participants : 6
Messages : 3174
System Messages : 4
Media Shared : 32
Deleted Messages : 15
Unique Words : 552


PART - 4 THE FINAL REPORT

In [37]:
print("="*75)
print("MESSAGE DISTRIBUTION")
print("="*75)

highest = max(message_count.values())

for user in sorted(participants):

    count = message_count[user]

    length = int((count/highest)*50)

    bar = "█"*length

    print(f"{user:10} | {bar} ({count})")

MESSAGE DISTRIBUTION
Aman       | █████████████████████████ (490)
Karan      | ██████████████████ (354)
Neha       | █████████████████████████████████ (635)
Priya      | █████████████████████████████████████ (718)
Rahul      | ██████████████████████████████████████████████████ (953)
Vikas      | █ (24)


In [38]:
print("="*75)
print("HOURLY ACTIVITY")
print("="*75)

highest = max(hour_count.values())

for hour in range(24):

    count = hour_count[hour]

    length = int((count/highest)*40)

    bar = "█"*length

    print(f"{hour:02d}:00 | {bar} {count}")

HOURLY ACTIVITY
00:00 | █████████ 57
01:00 | █████████████ 82
02:00 | █████████████ 83
03:00 | ████████████ 77
04:00 | █████████████████ 110
05:00 | ████ 29
06:00 | █████ 33
07:00 | ████████ 55
08:00 | ███████████████████ 122
09:00 | ████████████████████████ 151
10:00 | █████████████████████████ 160
11:00 | ██████████████████ 114
12:00 | ███████████████████████████████ 193
13:00 | █████████████████████████ 159
14:00 | ██████████████████████████ 162
15:00 | █████████████████████ 131
16:00 | ██████████████████████████████ 189
17:00 | ███████████████████████████ 173
18:00 | ████████████████████████████████████████ 248
19:00 | ████████████████████████████████████ 228
20:00 | ██████████████████████████ 166
21:00 | ████████████████████████████ 177
22:00 | ██████████████████ 116
23:00 | █████████████████████████ 159


In [39]:
days = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

highest = max(weekday_count.values())

print("="*75)
print("WEEKDAY ACTIVITY")
print("="*75)

for day in days:

    count = weekday_count[day]

    length = int((count/highest)*40)

    print(f"{day:10} {'█'*length} {count}")

WEEKDAY ACTIVITY
Monday     ███████████████████████████████████████ 482
Tuesday    ███████████████████████████████████ 426
Wednesday  ████████████████████████████████████████ 483
Thursday   ███████████████████████████████████████ 475
Friday     █████████████████████████████████ 403
Saturday   ██████████████████████████████████████ 459
Sunday     ████████████████████████████████████ 446


In [40]:
print("="*75)
print("TOP 25 WORDS")
print("="*75)

for word,count in counter.most_common(25):

    bar = "█"*(count//5)

    print(f"{word:20} {bar} {count}")

TOP 25 WORDS
the                  ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████ 1494
and                  ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████ 1013
was                  █████████████████████████████████████████████████████████████████████████████ 385
how                  ████████████████████████████████████████████████████████████████ 321
guys                 ███████████████████████████████████████████████████████████████ 318
you                  ██████████████████████████████████████████████████████████████ 312
this                 ██████████████████████████████████████████

In [45]:
print("="*70)
print('GROUPDNA REPORT  -  "Hostel Bois 4ever"')
print(f"{len(daily_messages)} days  •  {len(messages):,} messages  •  {len(participants)} members")
print("="*70)

print()

start_date=min(daily_messages.keys())
end_date=max(daily_messages.keys())

busiest_day=max(daily_messages,key=daily_messages.get)

busiest_hour=max(hour_count,key=hour_count.get)

print(f"Period          : {start_date} to {end_date}")
print(f"Busiest Day     : {busiest_day} ({daily_messages[busiest_day]} messages)")
print(f"Busiest Hour    : {busiest_hour:02d}:00 - {busiest_hour+1:02d}:00")

print()

print("="*70)
print("MESSAGES PER PERSON")
print("="*70)

highest=max(message_count.values())

for user,count in sorted(message_count.items(),
                         key=lambda x:x[1],
                         reverse=True):

    percent=(count/len(messages))*100

    bar="█"*int((count/highest)*25)

    print(f"{user:<10} {bar:<26} {count:>4} ({percent:.1f}%)")

GROUPDNA REPORT  -  "Hostel Bois 4ever"
60 days  •  3,174 messages  •  6 members

Period          : 2024-04-01 to 2024-05-30
Busiest Day     : 2024-05-04 (76 messages)
Busiest Hour    : 18:00 - 19:00

MESSAGES PER PERSON
Rahul      █████████████████████████   953 (30.0%)
Priya      ██████████████████          718 (22.6%)
Neha       ████████████████            635 (20.0%)
Aman       ████████████                490 (15.4%)
Karan      █████████                   354 (11.2%)
Vikas                                   24 (0.8%)


In [46]:
print()
print("="*70)
print("ACTIVITY HEATMAP")
print("="*70)

print("        00 03 06 09 12 15 18 21")

for user in sorted(participants):

    row=user.ljust(10)

    for h in [0,3,6,9,12,15,18,21]:

        count=0

        for msg in messages:

            if msg["sender"]==user and h<=msg["hour"]<h+3:

                count+=1

        if count>60:
            cell="█"

        elif count>30:
            cell="▓"

        elif count>10:
            cell="▒"

        elif count>0:
            cell="░"

        else:
            cell="."

        row+="   "+cell

    if personality[user]=="Night Owl":
        row+="   <- NIGHT OWL"

    print(row)


ACTIVITY HEATMAP
        00 03 06 09 12 15 18 21
Aman         █   █   .   .   ▒   ▓   ▓   █   <- NIGHT OWL
Karan        .   .   ▒   ▓   █   █   █   ▓
Neha         .   ▒   ▓   █   █   █   █   █
Priya        .   .   █   █   █   █   █   ▓
Rahul        ▓   ▓   ▓   ▓   █   █   █   █
Vikas        .   .   ░   ░   ░   ░   ░   ░


In [47]:
print()

print("="*70)
print("THIS GROUP'S FAVOURITE WORDS")
print("="*70)

top=counter.most_common(5)

highest=top[0][1]

for word,count in top:

    bar="█"*int((count/highest)*25)

    print(f"{word:<15}{bar:<27}{count}")


THIS GROUP'S FAVOURITE WORDS
the            █████████████████████████  1494
and            ████████████████           1013
was            ██████                     385
how            █████                      321
guys           █████                      318


In [48]:
print()

print("="*70)
print("RESPONSE PATTERNS")
print("="*70)

fastest=min(night_percentage,key=night_percentage.get)

slowest=max(night_percentage,key=night_percentage.get)

print(f"Fastest Replier : {fastest}")

print(f"Slowest Replier : {slowest}")


RESPONSE PATTERNS
Fastest Replier : Priya
Slowest Replier : Aman


In [49]:
print()

print("="*70)
print("LONGEST SILENT STREAKS")
print("="*70)

for user in sorted(participants):

    print(f"{user:<10}: {60-len(active_dates[user])} inactive days")


LONGEST SILENT STREAKS
Aman      : 0 inactive days
Karan     : 0 inactive days
Neha      : 0 inactive days
Priya     : 0 inactive days
Rahul     : 0 inactive days
Vikas     : 44 inactive days


In [53]:
print()

print("="*70)
print("PERSONALITY REPORT")
print("="*70)

for user in sorted(participants):

    print(f"{user:<10} : {personality[user]}")


PERSONALITY REPORT
Aman       :  Night Owl
Karan      : Story Teller
Neha       :  Balanced Human
Priya      :  Balanced Human
Rahul      :  Spammer
Vikas      :  Ghost
